
# Task 2: Time-Series Regression (Remaining Useful Life/RUL)

## 1. Project Overview

### 1.1 Problem Definition

This project focuses on predicting the **Remaining Useful Life (RUL)** of press machines using time‑series sensor measurements collected throughout operational cycles. 

The primary objective is to estimate the number of remaining operating cycles before machine failure occurs, enabling valuable information for maintenance scheduling and reliability management. 

To improve model practicality and stability, the target variable is capped at a **maximum value of one hundred fifty operating cycles**.

$$
RUL_t = \min(T_{\text{failure}} - t, 150)$$

### 1.2 Project Objectives

1. Perform Exploratory Data Analysis (EDA).
2. Conduct data preprocessing and RUL label generation.
3. Transform time-series data into sliding windows.
4. Develop and compare sequence-based Deep Learning models.
5. Establish a non-Deep Learning baseline.
6. Perform machine-level validation.
7. Estimate prediction uncertainty.
8. Evaluate models using RMSE and Asymmetric Score.

### 1.3 Evaluation Metrics

- Root Mean Squared Error (RMSE)
- Asymmetric Score

### 1.4 Project Requirements

- At least two sequence-based Deep Learning architectures.
- One non-Deep Learning baseline.
- Machine-level train-validation split.
- Prediction intervals using Monte Carlo Dropout or Ensemble.
- Final RUL prediction for test machines.



## 2. Library Imports

Several Python libraries are imported to facilitate data processing, file operations, archive handling, and formatted output generation, forming the foundation for subsequent project implementations:

- **Pandas:** Used for data manipulation and tabular data analysis.
- **Pathlib:** Used to manage file and directory paths.
- **IPython.display:** Provides functions to display formatted outputs and render Markdown content within the notebook.
- **ZipFile:** Used to access and read files directly from a ZIP archive without manually extracting its contents.

Additional libraries will be imported in subsequent sections depending on the requirements of data analysis, preprocessing, visualization, and model development.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display, Markdown
from zipfile import ZipFile

## 3. Dataset Understanding

### 3.1 Dataset Archive Inventory

The following code inspects the compressed dataset archive and builds a formatted summary table displaying each file name along with its corresponding storage size in megabytes.

In [24]:
zip_path = Path(r"C:\Users\LENOVO\OneDrive\Documents\Telkom University\SEMESTER 9\Deep Learning\UTS\data_uts_dl.zip")

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_ref: #r means read, zip_ref is for the zip folder that opened
        files = [item for item in zip_ref.infolist() if not item.is_dir()] #infolist takes zip lists, dir represent if the item a file not a dictionary
        inventory = pd.DataFrame([{
        "No": idx + 1,
        "File Name": file.filename,
        "File Size (MB)": file.file_size / (1024 * 1024) #byte to mb
        } for idx, file in enumerate(files)])

    display(
        inventory.style
        .set_caption("Dataset Inventory")
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("text-align", "center"),
                    ("font-size", "14px"),
                    ("font-weight", "bold"),
                    ("caption-side", "top"),
                    ("padding-bottom", "8px")
                ]},
            {"selector": "th", "props": [("text-align", "center")]}
        ])
        .format({"File Size (MB)": "{:.2f}"})
        .set_properties(**{"text-align": "center"})
        .set_properties(subset=["File Name"], **{"text-align": "left"})
        .hide(axis="index")
    )

else: 
    print("Archive data_uts_dl.zip not found at local path. Proceeding with data/ folder.")

No,File Name,File Size (MB)
1,klasifikasi_train_meta.csv,1.73
2,klasifikasi_test.npz,58.27
3,kamus_data.md,0.00
4,klasifikasi_train.npz,170.99
5,rul_test.csv.gz,3.94
6,rul_train.csv.gz,19.40
7,clustering_meta.csv,0.57
8,klasifikasi_test_meta.csv,0.46
9,clustering.npz,124.22


### 3.2 Data Dictionary Inspection

The following code extracts and renders the official data dictionary from the archive so that all variable definitions and target labeling rules remain clearly documented in the notebook.

In [26]:
kamusData = (Path.cwd().parent if Path.cwd().name.lower() == "kode" else Path.cwd()) / "kamus_data.md"

if zip_path.exists():
    with ZipFile(zip_path, "r") as zip_ref:
        kamus_data = next(
            name for name in zip_ref.namelist()
            if name.lower().endswith("kamus_data.md")
        )
        kamus_text = zip_ref.read(kamus_data).decode("utf-8-sig", errors="replace")
    display(Markdown(kamus_text))
elif kamusData.exists():
    display(Markdown(kamusData.read_text(encoding="utf-8")))
else:
    print("kamus_data.md not found locally.")

# Kamus data UTS Deep Learning

Semua kelompok menerima data yang sama. Label data uji tidak dibagikan.

## Tugas 1: klasifikasi_train.npz, klasifikasi_test.npz, klasifikasi_*_meta.csv

Satu sampel per unit produk dari pabrik komponen fiktif. Baris ke-i di berkas meta
memakai `indeks` = i pada array di berkas npz.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), foto permukaan produk |
| `sensor` | Array float16 (N, 3, 128): suhu (C), getaran (relatif), arus (A) selama penekanan. NaN berarti data tidak terekam |
| id_unit | ID unit produk |
| lini | Lini produksi (A atau B) |
| shift | Shift produksi |
| id_mesin | Mesin penekan |
| batch_material | Batch bahan baku |
| kelas | Label: normal, gores, noda, retak, gelembung, las_dingin, kontaminasi. Hanya di data latih |

Label data latih berasal dari inspektur dan tidak dijamin bebas salah.

## Tugas 2: rul_train.csv.gz dan rul_test.csv.gz

Log per siklus untuk mesin press fiktif. Data latih berisi riwayat sampai mesin rusak.
Data uji berhenti pada titik acak sebelum rusak.

| Kolom | Keterangan |
| --- | --- |
| id_unit | ID mesin |
| siklus | Nomor siklus sejak mesin mulai dipakai |
| setel_1, setel_2, setel_3 | Pengaturan operasi pada siklus itu |
| s01-s21 | Bacaan sensor. NaN berarti bacaan hilang |

Label: sisa umur pakai (RUL) = jumlah siklus tersisa sampai rusak, dipotong pada 150
(sisa umur di atas 150 siklus dihitung 150). Di data latih, siklus terakhir tiap unit adalah
siklus saat rusak, jadi labelnya dihitung sendiri. Prediksi RUL dibuat pada siklus terakhir
tiap unit di data uji.

## Tugas 3: clustering.npz dan clustering_meta.csv

Sampel material dari laboratorium fiktif, tanpa label.

| Isi | Keterangan |
| --- | --- |
| `gambar` | Array uint8 (N, 64, 64), citra mikrostruktur dari mikroskop |
| `spektrum` | Array float16 (N, 128), spektrum komposisi |
| id_sampel | ID sampel |
| mikroskop | Mikroskop yang dipakai (M1 atau M2) |


### 3.3 Loading and Initial Dataset Verification

The RUL training and testing datasets are loaded into Pandas DataFrames. Their dimensions, column structures, and machine identifiers are examined to ensure data integrity before proceeding to exploratory data analysis.

In [29]:
project_dir = Path.cwd()
if project_dir.name.lower() == "kode":
    project_dir = project_dir.parent

data_dir = project_dir / "data"
data_dir.mkdir(parents=True, exist_ok=True) #data dictionary

with ZipFile(zip_path, "r") as zip_ref:
    files = zip_ref.namelist()
    train_file = next(name for name in files if name.lower().endswith("rul_train.csv.gz"))
    test_file = next(name for name in files if name.lower().endswith("rul_test.csv.gz"))

    train_path = data_dir / "rul_train.csv.gz"
    test_path = data_dir / "rul_test.csv.gz"

    train_path.write_bytes(zip_ref.read(train_file))
    test_path.write_bytes(zip_ref.read(test_file))

train = pd.read_csv(train_path, compression="gzip")
test = pd.read_csv(test_path, compression="gzip")

print(f"Training shape    : {train.shape}")
print(f"Testing shape     : {test.shape}")
print(f"Training machines : {train['id_unit'].nunique()}")
print(f"Testing machines  : {test['id_unit'].nunique()}")

expected_columns = ["id_unit", "siklus", "setel_1", "setel_2", "setel_3"] + [f"s{i:02d}" for i in range(1, 22)]
print("Missing train cols:", [c for c in expected_columns if c not in train.columns])
print("Missing test cols :", [c for c in expected_columns if c not in test.columns])

display(
    train.head(3).style
    .set_caption("Table 3.1 Sample Records of RUL Training Dataset")
    .set_table_styles([
        {"selector": "caption", "props": [("text-align", "center"), ("font-weight", "bold"), ("caption-side", "top")]},
        {"selector": "th", "props": [("text-align", "center")]}
    ])
    .set_properties(**{"text-align": "center"})
    .hide(axis="index")
)

Training shape    : (254174, 26)
Testing shape     : (51692, 26)
Training machines : 900
Testing machines  : 300
Missing train cols: []
Missing test cols : []


id_unit,siklus,setel_1,setel_2,setel_3,s01,s02,s03,s04,s05,s06,s07,s08,s09,s10,s11,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
3,1,20.001000,0.700500,100.000000,1107.746000,569.289000,991.700000,1099.533000,nan,1313.240000,725.325000,778.517000,499.195000,782.886000,1495.906000,1092.248000,1144.779000,1345.634000,873.087000,851.099000,408.142000,997.673000,838.331000,1012.076000,439.003000
3,2,34.996000,0.842400,100.000000,1071.040000,587.821000,1013.673000,nan,157.684000,1304.796000,729.707000,776.236000,452.236000,819.193000,1492.101000,1095.511000,1167.320000,1340.661000,871.230000,791.396000,398.338000,980.601000,814.985000,1108.741000,412.368000
3,3,0.001000,-0.001200,100.000000,1130.132000,585.094000,998.221000,1127.436000,161.918000,1314.296000,725.847000,783.364000,533.958000,829.940000,1505.026000,1091.245000,1170.340000,1330.785000,872.114000,807.389000,388.304000,938.802000,847.512000,1003.616000,455.443000
